# 🤖 Megaman RL Training Facility (The Dojo)

### 📋 Instructions
1. **Runtime:** Ensure you are using **T4 GPU** (Runtime -> Change runtime type).
2. **Run All:** Execute the cells below in order.
3. **Auth:** You will be asked to authorize Google Drive access for saving checkpoints.

In [ ]:
# @title 1. Initialize & Mount Drive
from google.colab import drive
import os

print('💾 Mounting Google Drive...')
drive.mount('/content/drive')

DRIVE_LOGS_DIR = '/content/drive/MyDrive/Megaman_Training_Logs'
os.makedirs(DRIVE_LOGS_DIR, exist_ok=True)
print(f'✅ Checkpoints will save to: {DRIVE_LOGS_DIR}')

REPO_URL = 'https://github.com/LoganStunts/Megaman-RL-Dojo.git'
REPO_NAME = 'Megaman-RL-Dojo'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd /content/{REPO_NAME}
    !git pull
    %cd ..

%cd /content/{REPO_NAME}
print('✅ Repo Ready.')

In [ ]:
# @title 2. Infrastructure Setup
!apt-get install -y libxcursor1 libxinerama1 libxrandr2 libxi6 libgl1-mesa-dev xvfb
!pip install godot-rl stable-baselines3 shimmy>=0.2.1 tensorboard huggingface_sb3 onnx

if not os.path.exists('Godot_v4.5.1-stable_linux.x86_64'):
    !wget https://github.com/godotengine/godot/releases/download/4.5.1-stable/Godot_v4.5.1-stable_linux.x86_64.zip
    !unzip -o Godot_v4.5.1-stable_linux.x86_64.zip
    !chmod +x Godot_v4.5.1-stable_linux.x86_64
print('✅ Infrastructure Ready.')

In [ ]:
# @title 3. Select Skill (Dynamic Branch Switching)
import requests
from google.colab import widgets

# 1. Fetch Branches from GitHub
repo_api_url = "https://api.github.com/repos/LoganStunts/Megaman-RL-Dojo/branches"
response = requests.get(repo_api_url)
branches = [b['name'] for b in response.json() if b['name'].startswith('payload/')]
if not branches: branches = ['payload/main']

# 2. Colab Form UI
SELECTED_BRANCH = "payload/main" # @param ['payload/main'] {allow-input: true}

# Update logic
print(f"🎯 Skill Selected: {SELECTED_BRANCH}")
!git fetch origin
!git checkout {SELECTED_BRANCH}
!git pull origin {SELECTED_BRANCH}
print(f"✅ Dojo configured for: {SELECTED_BRANCH}")

In [ ]:
# @title 4. Ignite Training (Optimized)
import subprocess, time
subprocess.run(['pkill', '-f', 'Godot'])
time.sleep(2)

GODOT_BIN = '/content/Megaman-RL-Dojo/Godot_v4.5.1-stable_linux.x86_64'

print('📦 Importing assets...')
with open('godot_import.log', 'w') as f:
    subprocess.run(['xvfb-run', '-a', GODOT_BIN, '--headless', '--editor', '--quit', '--path', '/content/Megaman-RL-Dojo'], stdout=f, stderr=f)

print('🧠 Starting Optimized Trainer...')
!xvfb-run -a python3 train_optimized.py --env_path={GODOT_BIN} --n_parallel=4 --speedup=4 --save_path={DRIVE_LOGS_DIR} --timesteps=1000000

In [ ]:
# @title 5. Secure Uplink (Auto-Push Results)
# Ensure you have set your GITHUB_TOKEN in the Colab Secrets (Key icon on the left)
from google.colab import userdata
import os

try:
    # Configure Auth
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_URL_AUTH = f'https://{GITHUB_TOKEN}@github.com/LoganStunts/Megaman-RL-Dojo.git'
    
    # Update Remote
    !git remote set-url origin {REPO_URL_AUTH}
    
    # Ignite Upload Script
    !python3 dojo_upload.py
    
except Exception as e:
    print("\u26a0\ufe0f Uplink Failed: Is GITHUB_TOKEN set in Secrets?")
    print(e)